# Dispatch Eval — Exploratory Analysis

Loads `results.jsonl`, `runs_meta.jsonl`, and `traces.jsonl` from `../` (the eval directory) into pandas for interactive analysis.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

EVAL_DIR = Path("..").resolve()   # evals/intelligence_agent/

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_colwidth", 80)

# Sanity check — catch wrong working directory early
for fname in ["results.jsonl", "runs_meta.jsonl", "traces.jsonl"]:
    p = EVAL_DIR / fname
    assert p.exists(), f"Expected {p} — is the kernel cwd the notebook directory?"
print(f"EVAL_DIR: {EVAL_DIR}  ✓")

## Load and flatten results

In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

results_raw = load_jsonl(EVAL_DIR / "results.jsonl")
runs_raw    = load_jsonl(EVAL_DIR / "runs_meta.jsonl")

# Flatten nested dicts (assertions.*, tuple.*, inputs_meta.anomaly_config.*)
df = pd.json_normalize(results_raw, sep=".")

# Shorten column names for convenience
df = df.rename(columns=lambda c: c.replace("assertions.", "a.").replace("tuple.", ""))

# tuple.id renames to id, colliding with the top-level id — drop the duplicate
df = df.loc[:, ~df.columns.duplicated()]

# Boolean columns: True=pass, False=fail, None=N/A
assertion_cols = [c for c in df.columns if c.startswith("a.")]

print(f"{len(df)} result rows  |  {df['run_id'].nunique()} runs  |  {df['id'].nunique()} tuples")
df.head()

In [ ]:
# Load runs metadata and join in model + threshold info
runs_df = pd.json_normalize(runs_raw, sep=".").rename(
    columns=lambda c: c.replace("inputs_meta.anomaly_config.", "cfg.").replace("inputs_meta.", "")
)
runs_df = runs_df.set_index("run_id")

# Join thresholds and token counts — model is already in results.jsonl, skip it to avoid collision
df = df.join(runs_df[["cfg.zscore_threshold", "cfg.drift_threshold",
                       "cfg.focus_decline_pct", "total_input_tokens", "total_output_tokens"]],
             on="run_id")

runs_df[["model", "n_inputs", "total_input_tokens", "total_output_tokens",
         "cfg.zscore_threshold", "cfg.drift_threshold", "cfg.focus_decline_pct"]]

## Overall pass rates by assertion

In [ ]:
# Mean pass rate per assertion (ignoring N/A)
pass_rates = df[assertion_cols].apply(lambda col: col.dropna().astype(float).mean())
pass_rates.index = pass_rates.index.str.replace("a.", "", regex=False)

fig, ax = plt.subplots(figsize=(8, 4))
pass_rates.sort_values().plot.barh(ax=ax, color="steelblue")
ax.axvline(1.0, color="green", linestyle="--", linewidth=1)
ax.set_xlabel("Pass rate")
ax.set_title("Overall pass rate by assertion")
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.show()

## Pass rates by model

In [ ]:
by_model = (
    df.groupby("model")[assertion_cols]
    .apply(lambda g: g.apply(lambda col: col.dropna().astype(float).mean()))
)
by_model.columns = by_model.columns.str.replace("a.", "", regex=False)

by_model.T.plot.barh(figsize=(9, 5))
plt.xlabel("Pass rate")
plt.title("Pass rate by assertion × model")
plt.xlim(0, 1.05)
plt.legend(title="model", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## Pass rates by anomaly_type and severity

In [ ]:
for col in assertion_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

pivot = (
    df.groupby(["anomaly_type", "severity"])[assertion_cols]
    .mean()
    .round(2)
)
pivot.columns = pivot.columns.str.replace("a.", "", regex=False)
pivot

In [ ]:
# Heatmap: anomaly_type × assertion, mean pass rate
heat = df.groupby("anomaly_type")[assertion_cols].mean()
heat.columns = heat.columns.str.replace("a.", "", regex=False)

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("Pass rate by anomaly_type × assertion")
plt.tight_layout()
plt.show()

## Pass rates by event_context

In [ ]:
heat_ctx = df.groupby("event_context")[assertion_cols].mean()
heat_ctx.columns = heat_ctx.columns.str.replace("a.", "", regex=False)

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(heat_ctx, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("Pass rate by event_context × assertion")
plt.tight_layout()
plt.show()

## Consistently failing tuples

Tuples that fail the same assertion across multiple runs — good candidates for deeper review.

In [ ]:
# Ensure assertion columns are numeric (True→1.0, False→0.0, None→NaN)
# regardless of cell execution order
for col in assertion_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# tuple_rates: mean PASS RATE per (tuple, assertion) across all runs.
#   1.0 = passed every run it was applicable
#   0.0 = failed every run it was applicable
#   NaN = assertion was never applicable for this tuple (e.g. critical_urgent on a warn tuple)

# MIN_PASS_RATE: lower = stricter filter, shows only the worst offenders.
#   1.0 → show every tuple that failed even once
#   0.5 → show only tuples failing MORE than half the time
#   0.0 → show nothing (every tuple passes at least 0% of the time)
MIN_PASS_RATE = 0.6

tuple_rates = (
    df.groupby(["id", "anomaly_type", "severity", "event_context"])[assertion_cols]
    .mean()
    .round(2)
)
tuple_rates.columns = tuple_rates.columns.str.replace("a.", "", regex=False)

struggling = tuple_rates[(tuple_rates < MIN_PASS_RATE).any(axis=1)]

print(f"Tuples with at least one assertion passing fewer than {int(MIN_PASS_RATE*100)}% of runs:")
print(f"  {len(struggling)} of {len(tuple_rates)} tuples")
print("Values are PASS RATES (0.0=always fails, 1.0=always passes, blank=N/A)\n")
struggling

## Inspect raw responses for a specific tuple

In [ ]:
TUPLE_ID = 14  # change this to the id you want to inspect

traces_raw = load_jsonl(EVAL_DIR / "traces.jsonl")
traces_df = pd.json_normalize(traces_raw, sep=".")

subset = traces_df[traces_df["id"] == TUPLE_ID][["run_id", "model", "response", "error"]]
for _, row in subset.iterrows():
    print(f"--- run={row['run_id']}  model={row.get('model', '?')} ---")
    print(row["response"])
    print()